# Experiment 5 — ICP Hyperparameter Optimization

**Goal:** Find the best ICP configuration per cloud geometry using Optuna multi-objective optimization.

Two objectives are minimized jointly:
- Mean rotation error vs ground truth (°)
- Mean translation error vs ground truth

Each trial evaluates a configuration over `N_SEEDS` random seeds.
A separate study is run per cloud style: `random`, `clustered`, `lattice`.

`max_iter` and `tol` are fixed — they are stopping criteria, not quality parameters.

In [ ]:
import sys
sys.path.insert(0, '../src')

import numpy as np
import matplotlib.pyplot as plt
import optuna
from tabulate import tabulate
from optuna.importance import get_param_importances

from hpo import make_objective
from visualization import OptimizationVisualizer

optuna.logging.set_verbosity(optuna.logging.WARNING)
plt.style.use('seaborn-v0_8-whitegrid')

STYLES     = ['random', 'clustered', 'lattice']
N_SEEDS    = 15
N_TRIALS   = 50
MAX_ITER   = 400
TOL        = 1e-10
GEN_KWARGS = dict(n=2000, noise_std=0.0, t_scale=8.0)

## 1. Run Studies

One multi-objective Optuna study per cloud style. NSGA-II sampler explores the joint space of matching strategy and soft-matching hyperparameters.

In [2]:
studies = {}
for style in STYLES:
    print(f'Running study for style={style!r} ...')
    sampler = optuna.samplers.NSGAIISampler(seed=0)
    study = optuna.create_study(
        directions=['minimize', 'minimize'],
        sampler=sampler,
        study_name=f'icp_hpo_{style}',
    )
    study.optimize(
        make_objective(style, N_SEEDS, GEN_KWARGS, MAX_ITER, TOL),
        n_trials=N_TRIALS,
        show_progress_bar=True,
    )
    studies[style] = study
    print(f'  Done. Pareto front size: {len(study.best_trials)}')

Running study for style='random' ...


100%|██████████| 50/50 [21:38<00:00, 25.97s/it] 


  Done. Pareto front size: 3
Running study for style='clustered' ...


100%|██████████| 50/50 [20:13<00:00, 24.27s/it] 


  Done. Pareto front size: 7
Running study for style='lattice' ...


100%|██████████| 50/50 [15:03<00:00, 18.07s/it]

  Done. Pareto front size: 2


## 2. Pareto Fronts

Each point is one trial. Pareto-optimal trials (red outline) are not dominated on either objective.
Points are coloured by matching strategy.

In [ ]:
COLOR = {'hard': 'tab:blue', 'soft': 'tab:orange'}
pareto_numbers = {style: {t.number for t in studies[style].best_trials} for style in STYLES}

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for ax, style in zip(axes, STYLES):
    study = studies[style]
    trials = [t for t in study.trials if t.values is not None]

    x_values    = np.array([t.values[0] for t in trials])
    y_values    = np.array([t.values[1] for t in trials])
    group_ids   = [t.params.get('matching', 'unknown') for t in trials]
    pareto_mask = np.array([t.number in pareto_numbers[style] for t in trials])

    OptimizationVisualizer.plot_pareto_front(
        ax, x_values, y_values, group_ids, pareto_mask,
        colors=COLOR,
        x_label='Mean rotation error (°)',
        y_label='Mean translation error',
    )
    ax.set_title(f"style='{style}'")
    ax.legend(title='matching')

fig.suptitle('Pareto Fronts — rotation vs translation error (red outline = Pareto-optimal)')
plt.tight_layout()
plt.savefig('../results/5_icp_hyperparam_opt_trial_scatter.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Best Configurations

For each style, the Pareto-optimal trial closest to the origin in normalized objective space
is selected as the representative best configuration.

In [4]:
for style in STYLES:
    study = studies[style]
    all_vals = np.array([t.values for t in study.trials])
    val_min, val_max = all_vals.min(axis=0), all_vals.max(axis=0)
    norm = (val_max - val_min) + 1e-9

    pareto_trials = study.best_trials
    pareto_norm = (np.array([t.values for t in pareto_trials]) - val_min) / norm
    best = pareto_trials[np.linalg.norm(pareto_norm, axis=1).argmin()]

    rows = [
        ['Rotation error (°)', f"{best.values[0]:.3f}"],
        ['Translation error',  f"{best.values[1]:.3f}"],
        ['Reliability',        f"{best.user_attrs['reliability']:.0%}"],
    ] + [[k, f"{v:.4g}" if isinstance(v, float) else v] for k, v in best.params.items()]

    sigma_final = best.user_attrs.get('sigma_final')
    if sigma_final is not None:
        rows.append(['sigma_final (derived)', f'{sigma_final:.4g}'])

    print(f"\n=== Best config for style='{style}' ===")
    print(tabulate(rows, headers=['Parameter', 'Value'], tablefmt='rounded_outline'))


=== Best config for style='random' ===
╭───────────────────────┬─────────╮
│ Parameter             │ Value   │
├───────────────────────┼─────────┤
│ Rotation error (°)    │ 103.714 │
│ Translation error     │ 16.137  │
│ Reliability           │ 13%     │
│ matching              │ soft    │
│ sigma_init            │ 1.078   │
│ sigma_ratio           │ 0.03254 │
│ anneal_steps          │ 281     │
│ k                     │ 12      │
│ sigma_final (derived) │ 0.03507 │
╰───────────────────────┴─────────╯

=== Best config for style='clustered' ===
╭───────────────────────┬─────────╮
│ Parameter             │ Value   │
├───────────────────────┼─────────┤
│ Rotation error (°)    │ 97.322  │
│ Translation error     │ 14.354  │
│ Reliability           │ 20%     │
│ matching              │ soft    │
│ sigma_init            │ 1.078   │
│ sigma_ratio           │ 0.03254 │
│ anneal_steps          │ 281     │
│ k                     │ 12      │
│ sigma_final (derived) │ 0.03507 │
╰────────────────

## 4. Parameter Importance

Importance is computed separately for each objective using **PedAnova**, which handles the conditional parameter space (soft-matching params are only present in soft-matching trials).

In [ ]:
OBJ_LABELS = ['Rotation error (°)', 'Translation error']
ALL_PARAMS = ['matching', 'sigma_init', 'sigma_ratio', 'anneal_steps', 'k']

fig, axes = plt.subplots(len(STYLES), 2, figsize=(12, 4 * len(STYLES)), squeeze=False)

for row, style in enumerate(STYLES):
    study = studies[style]
    for col, (obj_idx, obj_label) in enumerate(zip([0, 1], OBJ_LABELS)):
        ax = axes[row, col]
        try:
            importance = get_param_importances(
                study,
                target=lambda t, i=obj_idx: t.values[i],
                evaluator=optuna.importance.PedAnovaImportanceEvaluator(),
                params=ALL_PARAMS,
            )
        except Exception as e:
            ax.text(0.5, 0.5, f'N/A\n{e}', ha='center', va='center', transform=ax.transAxes)
            ax.set_title(f"'{style}' — {obj_label}")
            continue

        OptimizationVisualizer.plot_feature_importance(ax, importance, title=f"'{style}' — {obj_label}")

fig.suptitle('Parameter importance per cloud style and objective')
plt.tight_layout()
plt.savefig('../results/5_icp_hyperparam_opt_feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()